# COP509 -- Memory-Efficient Model Loading (SOLUTION)

> Large language models rely on massive matrix multiplications that benefit enormously from GPU parallelism and memory bandwidth. While LLMs *can* run on CPUs, inference is impractically slow for most use cases -- so in practice, **you need a GPU**. But GPU memory is **scarce and expensive** (a T4 has just 16 GB, an A100 has 40--80 GB), which makes fitting a model into GPU memory one of the first practical challenges you face.
>
> **The problem:** Loading a model in PyTorch the standard way temporarily uses **~2x the model size in GPU memory**.  
> A 500 MB model briefly needs ~1 GB of GPU. A 1.4 GB model needs ~2.8 GB. A 6 GB model needs ~12 GB.  
> As models scale up, this 2x overhead becomes the difference between "fits on your GPU" and "crashes immediately."  
>  
> This is a limitation of PyTorch's default serialization format (.pt/.bin). Other approaches — such as alternative frameworks (TensorFlow) or alternative file formats (SafeTensors) — handle loading more efficiently.

In this lab you will use a **GPT-2 124M** model to observe this doubling effect first-hand, then learn **4 techniques** to eliminate it. The model is small enough to experiment safely on a T4 GPU (16 GB), but the techniques you learn here are **essential** for loading production-scale models (7B, 70B, and beyond) where the 2x overhead simply won't fit.

---

*Based on Sebastian Raschka's [Build a Large Language Model From Scratch](https://github.com/rasbt/LLMs-from-scratch) (Apache 2.0 License).*

## Setup

**Important:** Before running, make sure you have a GPU runtime:  
`Runtime -> Change runtime type -> T4 GPU`

Run the two cells below to install dependencies and import helpers.

In [ ]:
!pip install -q psutil matplotlib

In [ ]:
import os
# Uncomment the line below once _infrastructure.py is pushed to the GitHub repo:
# if not os.path.exists("_infrastructure.py"):
#     !wget -q https://raw.githubusercontent.com/gcosma/COP509/main/COP509-memory-lab/_infrastructure.py

# For now, upload _infrastructure.py manually to Colab (drag into the Files panel on the left)

import torch
import gc
from _infrastructure import (
    GPTModel, GPT_CONFIGS, MODEL_SIZES_MB,
    create_and_save_model,
    track_memory, MemoryTracker, cleanup,
    track_time, TimeTracker,
    plot_memory_comparison, print_memory_summary,
    plot_timing_comparison, print_timing_summary,
    get_device, has_gpu,
)

device = get_device()
tracker = MemoryTracker()
print(f"\nPyTorch version: {torch.__version__}")
print(f"Device: {device}")

---

## Task 1: Create & Save a Model

Before we can practise *loading*, we need a saved model file.

We will create a **GPT-2 124M** model with random weights and save it to disk. The weights are not trained, but that is fine -- our focus is on **how loading affects memory**, not on what the model predicts.

### Quick estimate: how big should this file be?

Each parameter is a 32-bit float = **4 bytes**.

> 124,000,000 params x 4 bytes = **496,000,000 bytes -- roughly 500 MB**

Keep that number in your head. Run the cell below and compare the **actual file size** to your estimate.

In [ ]:
# TODO 1 -- COMPLETED
create_and_save_model("124M")

### 🛑 Stop & Think #1

The file is bigger than 500 MB. Why?

<details>
<summary><b>Click for answer</b></summary>

<br>

The "124M" figure refers to the **core transformer parameters** -- the weights in the 12 attention layers and feed-forward networks. But a complete model also includes:

- **Token embeddings:** 50,257 vocabulary size x 768 hidden dimension = approximately 38.6M parameters
- **Position embeddings:** 1,024 positions x 768 = approximately 0.8M parameters
- **Output projection head:** 768 x 50,257 = approximately 38.6M parameters
- **Layer norm parameters** in each block

When you include these, the total is closer to **approximately 163 million parameters**. OpenAI used "124M" as shorthand to describe the model's architectural size class (small / medium / large / XL), not as an exact parameter count.

So: 163M params x 4 bytes = approximately 652 MB, plus `torch.save()` serialisation overhead, giving approximately **670 MB**.

**Takeaway:** Model names are marketing labels, not precise specs. Always check the actual architecture if you need to estimate memory.

</details>

---

## Task 2: Standard Loading (The Baseline)

Most PyTorch tutorials load a model like this:

```
Step 1: model = GPTModel(config)        -- allocate memory for the model
Step 2: model.to(device)                -- move to GPU
Step 3: checkpoint = torch.load(...)    -- load weights into a dictionary
Step 4: model.load_state_dict(...)      -- copy weights into the model
```

Seems simple enough. But watch what happens to memory when you run the cell below...

In [ ]:
# TODO 2 -- COMPLETED
with track_memory(tracker, "Standard"):
    model = GPTModel(GPT_CONFIGS["124M"]).to(device)            # ~670 MB allocated on GPU
    checkpoint = torch.load("gpt2-124M.pth", map_location=device, weights_only=True)  # +670 MB on GPU (both exist now!)
    model.load_state_dict(checkpoint)                           # copies checkpoint INTO model (peak memory here)
    del checkpoint                                              # frees the checkpoint -- but the peak already happened

In [ ]:
del model
cleanup()

### 🛑 Stop & Think #2

Look at the memory numbers above.

- The model is approximately **670 MB** on disk.
- But peak GPU memory was significantly **higher** than 670 MB.

**Why?** Where did the extra memory go?

<details>
<summary><b>Click for answer</b></summary>

<br>

When you call `model.load_state_dict(checkpoint)`, PyTorch **copies** the weights from the checkpoint dictionary into the model's parameters. During this copy, **both** exist in GPU memory at the same time -- that's roughly **2x the model size**.

After the copy is done, the checkpoint is no longer needed, but Python doesn't automatically free it -- it stays in memory until you explicitly run `del checkpoint` (or it goes out of scope). So the peak memory is always at least 2x, regardless of when you clean up.

For a 6 GB model, this means you temporarily need approximately 12 GB just to load it -- which may not fit on your GPU.

</details>

---

## Task 3: Sequential Parameter Loading

**Idea:** Instead of loading all weights at once on the GPU, what if we:
1. Load the checkpoint on **CPU** (cheaper memory)
2. Copy parameters to the GPU **one at a time**
3. Delete each parameter from the checkpoint as we go

This way, the GPU never holds the full checkpoint.

In [ ]:
# TODO 3 -- COMPLETED
with track_memory(tracker, "Sequential"):
    model = GPTModel(GPT_CONFIGS["124M"]).to(device)                # model on GPU (~670 MB)
    state_dict = torch.load("gpt2-124M.pth", map_location="cpu", weights_only=True)  # checkpoint on CPU, NOT GPU

    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name].to(device))            # copy one param at a time to GPU

    del state_dict                                                  # free CPU memory

In [ ]:
del model
cleanup()

### 🛑 Stop & Think #3

Compare the **peak GPU memory** to Task 2.

- Did GPU memory improve? By how much?
- What about CPU memory -- was it higher or lower than before?
- Why does this technique help GPU but not CPU?

<details>
<summary><b>Click for answer</b></summary>

<br>

**GPU memory is lower** because we loaded the checkpoint onto CPU, not GPU. The GPU only ever holds the model itself + one parameter being copied at a time.

**CPU memory is still high** because we loaded the entire checkpoint into RAM with `torch.load(... map_location="cpu")`. The full state_dict lives in CPU memory until we delete it.

This technique shifts the memory burden from GPU to CPU.

</details>

---

## Task 4: The Meta Device Trick

PyTorch has a special device called `"meta"`. When you create a model on meta, PyTorch builds the full model structure -- all the layers, all the connections -- and records the **shape and dtype** of every parameter, but allocates **zero bytes** of actual memory for the data.

Why is this useful? Because PyTorch needs to know the shape and dtype of each parameter **before** it can load weights into them. Normally, you get this by creating the model for real (which allocates all the memory). With `meta`, you get the same structural information for free -- no memory cost. Then you can fill in the actual data later, directly from the checkpoint, without ever having a redundant copy.

The plan:
1. Create the model on `meta` (0 bytes -- just structure)
2. Allocate real empty tensors on GPU with `to_empty()` 
3. Load weights with `assign=True` (swaps in the real data, no copy)

In [ ]:
# TODO 4 -- COMPLETED
with track_memory(tracker, "Meta Device"):
    with torch.device("meta"):
        model = GPTModel(GPT_CONFIGS["124M"])                       # 0 bytes! just a blueprint (no real memory)

    model = model.to_empty(device=device)                           # allocate empty tensors on GPU (~670 MB, no real data yet)

    state_dict = torch.load("gpt2-124M.pth", map_location=device, weights_only=True)  # load checkpoint directly onto GPU (device="cuda", NOT "meta")
    model.load_state_dict(state_dict, assign=True)                  # assign=True: checkpoint tensors REPLACE the empty ones (no copy, no duplication)
    del state_dict                                                  # nothing extra to free -- tensors are already the model's params

In [ ]:
del model
cleanup()

### 🛑 Stop & Think #4

Compare **peak CPU memory** across all three techniques so far.

- Why is CPU memory dramatically lower with the meta device?
- What does `assign=True` do differently from the default `load_state_dict()`?

<details>
<summary><b>Click for answer</b></summary>

<br>

With the default `load_state_dict()`, PyTorch **copies** each tensor from the checkpoint into the model's pre-allocated parameters. This means both exist in memory simultaneously.

With `assign=True`, PyTorch **directly assigns** the checkpoint's tensors as the model's parameters -- no copy, no duplication. Combined with the meta device (which never allocated real memory for the model), CPU usage plummets.

</details>

---

## Task 5: Memory-Mapped Loading (`mmap=True`)

This is the **recommended technique** for most real-world use.

Instead of reading the entire `.pth` file into RAM, `mmap=True` tells the OS to **map the file directly into virtual memory**. Data is read from disk **on demand**, page by page -- only what's needed gets loaded into physical RAM.

Think of it like streaming a video vs. downloading the whole thing first.

In [ ]:
# TODO 5 -- COMPLETED
with track_memory(tracker, "Memory Mapped"):
    with torch.device("meta"):
        model = GPTModel(GPT_CONFIGS["124M"])                       # 0 bytes -- blueprint only

    state_dict = torch.load(
        "gpt2-124M.pth", map_location=device,
        weights_only=True, mmap=True                                # mmap=True: read from disk on demand, not all into RAM
    )
    model.load_state_dict(state_dict, assign=True)                  # assign=True: no copy, direct swap
    del state_dict

In [ ]:
del model
cleanup()

### 🛑 Stop & Think #5

This is considered the **best practice** for loading large models.

- How does peak CPU memory compare to the standard approach?
- Why is `mmap=True` often preferred over the meta device trick alone?

<details>
<summary><b>Click for answer</b></summary>

<br>

`mmap=True` tells the OS to "memory-map" the file -- the data lives on disk, and only the pages you actually access get loaded into physical RAM. This reduces CPU memory usage compared to Standard loading.

It's preferred over the meta device trick (Task 4) because:
- **Simpler code** -- you don't need the `to_empty()` step
- **Better GPU memory** -- Task 4's meta device approach actually had the *worst* GPU memory because `to_empty()` allocates empty tensors on GPU before the checkpoint is also loaded onto GPU. Memory Mapped avoids this.
- **Best all-rounder** -- it ties for lowest GPU memory (with Sequential) while also being fast and requiring minimal code changes

Note: our Memory Mapped code still uses `with torch.device("meta")` and `assign=True` -- the `mmap=True` flag replaces the need for `to_empty()`, not the meta device itself.

</details>

---

## Task 6: The Big Comparison

Let's visualise all 4 techniques side by side.

In [ ]:
# TODO 6 -- COMPLETED
plot_memory_comparison(tracker)
print_memory_summary(tracker)

### 🛑 Stop & Think #6 -- Class Discussion

Look at the chart and discuss:

1. Which technique uses the **least peak GPU memory**?
2. Which technique uses the **least peak CPU memory**?
3. Which result surprises you the most?
4. If you had to pick **ONE technique** for production use, which would it be and why?
5. What are the **trade-offs**? (Code complexity? Speed? Compatibility?)

<details>
<summary><b>Click for answer</b></summary>

<br>

1. **Sequential** and **Memory Mapped** tie for the least peak GPU memory -- both around 1x the model size (~822 MB). They avoid having the full checkpoint and model on GPU simultaneously.
2. **Meta Device** uses the least peak CPU memory -- because the model is never fully allocated on CPU.
3. **Meta Device has the WORST GPU memory** -- even higher than Standard! This is because `to_empty()` allocates empty tensors on GPU, then the checkpoint is also loaded onto GPU. Even with `assign=True`, both exist during loading. Meta device helps CPU at the **expense** of GPU.
4. **Memory Mapped** is the best all-rounder for production -- it ties for lowest GPU, has reasonable CPU usage, is the **fastest** to load, and requires the least code changes.
5. Trade-offs: Sequential saves GPU but is one of the **slowest** (parameter-by-parameter loop). Meta Device slashes CPU but makes GPU **worse**. Memory Mapped is the best balance of memory, speed, and simplicity.

</details>

## Extension A: Scale Up -- Try GPT-2 355M

The 124M model is small. The memory savings become much more dramatic with larger models.

Download the 355M model (~1.4 GB) and load it with all 4 techniques. How do the savings scale?

In [ ]:
# TODO 7 -- COMPLETED
create_and_save_model("355M")

tracker_355 = MemoryTracker()

# 1. Standard loading
with track_memory(tracker_355, "Standard"):
    model = GPTModel(GPT_CONFIGS["355M"]).to(device)
    checkpoint = torch.load("gpt2-355M.pth", map_location=device, weights_only=True)
    model.load_state_dict(checkpoint)
    del checkpoint
del model
cleanup()

# 2. Sequential loading
with track_memory(tracker_355, "Sequential"):
    model = GPTModel(GPT_CONFIGS["355M"]).to(device)
    state_dict = torch.load("gpt2-355M.pth", map_location="cpu", weights_only=True)
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name].to(device))
    del state_dict
del model
cleanup()

# 3. Meta Device
with track_memory(tracker_355, "Meta Device"):
    with torch.device("meta"):
        model = GPTModel(GPT_CONFIGS["355M"])
    model = model.to_empty(device=device)
    state_dict = torch.load("gpt2-355M.pth", map_location=device, weights_only=True)
    model.load_state_dict(state_dict, assign=True)
    del state_dict
del model
cleanup()

# 4. Memory Mapped
with track_memory(tracker_355, "Memory Mapped"):
    with torch.device("meta"):
        model = GPTModel(GPT_CONFIGS["355M"])
    state_dict = torch.load(
        "gpt2-355M.pth", map_location=device,
        weights_only=True, mmap=True
    )
    model.load_state_dict(state_dict, assign=True)
    del state_dict
del model
cleanup()

plot_memory_comparison(tracker_355)
print_memory_summary(tracker_355)

### 🛑 Stop & Think -- Extension A

Look at the 355M chart compared to the 124M chart.

- Did the **gap** between Standard and Memory Mapped get bigger or smaller?
- If you tried this with the 1.5B model (~6 GB), would Standard loading even fit on the T4 GPU (16 GB)?
- What does this tell you about why these techniques matter more as models get larger?

<details>
<summary><b>Click for answer</b></summary>

The gap gets **much bigger** with larger models. Standard loading needs ~2x the model size in GPU memory, so for 355M that's ~3.3 GB on GPU vs ~1.8 GB for the efficient techniques.

For the 1.5B model (~6 GB), Standard loading would need ~12 GB just for loading -- leaving only 4 GB for everything else on a 16 GB T4. Memory-efficient techniques aren't just nice to have at that scale -- they're **essential**.

</details>

## Extension B: Individual File Loading (The Extreme Approach)

What if you saved **each parameter tensor as its own file**?  
Then you could load them one at a time with virtually zero overhead.

This is the most memory-efficient technique, but it creates many small files and is slower.

In [ ]:
# TODO 8 -- COMPLETED

# Step A: Save each parameter separately
state_dict = torch.load("gpt2-124M.pth", map_location="cpu", weights_only=True)
os.makedirs("gpt2-124M-params", exist_ok=True)
for name, param in state_dict.items():
    torch.save(param, f"gpt2-124M-params/{name}.pt")
del state_dict
gc.collect()

# Step B: Load them one at a time
with track_memory(tracker, "Individual Files"):
    with torch.device("meta"):
        model = GPTModel(GPT_CONFIGS["124M"])
    model = model.to_empty(device=device)

    with torch.no_grad():
        for name, param in model.named_parameters():
            data = torch.load(
                f"gpt2-124M-params/{name}.pt",
                map_location=device, weights_only=True
            )
            param.copy_(data)
            del data

# Step C: Updated comparison
plot_memory_comparison(tracker)
print_memory_summary(tracker)

### 🛑 Stop & Think -- Extension B

Look at the "5. Individual Files" bar compared to the other techniques.

- Is it the most memory-efficient? By how much?
- What are the downsides of saving hundreds of separate files?
- When would this approach actually be worth the complexity?

<details>
<summary><b>Click for answer</b></summary>

Individual file loading is the **most memory-efficient** -- it achieves essentially **zero CPU memory** because only one small tensor is ever loaded at a time, and GPU memory stays at ~1x the model size (same as Sequential and Memory Mapped).

The downsides are practical, not performance-related: hundreds of small files are messy to manage, harder to distribute (sharing a folder of 200+ files vs one `.pth` file), and add code complexity (you need a save step and a custom loading loop).

Speed-wise, it's actually **not as slow as you might expect** -- Extension C will show you exactly how it compares. The main reason to avoid it is the file management overhead, not speed. Use it when you need absolute minimum CPU memory and can tolerate the complexity.

</details>

## Extension C: Benchmark Loading Speed

Memory isn't the only thing that matters -- **how fast** does each technique load?

Is the most memory-efficient approach also the fastest? Or is there a trade-off?

> **Note:** This extension benchmarks all 5 techniques, including Individual Files from Extension B. Make sure you have run Extension B first.

In [ ]:
# TODO 9 -- COMPLETED
timer = TimeTracker()

# 1. Standard
with track_time(timer, "Standard"):
    model = GPTModel(GPT_CONFIGS["124M"]).to(device)
    checkpoint = torch.load("gpt2-124M.pth", map_location=device, weights_only=True)
    model.load_state_dict(checkpoint)
    del checkpoint
del model; cleanup()

# 2. Sequential
with track_time(timer, "Sequential"):
    model = GPTModel(GPT_CONFIGS["124M"]).to(device)
    state_dict = torch.load("gpt2-124M.pth", map_location="cpu", weights_only=True)
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name].to(device))
    del state_dict
del model; cleanup()

# 3. Meta Device
with track_time(timer, "Meta Device"):
    with torch.device("meta"):
        model = GPTModel(GPT_CONFIGS["124M"])
    model = model.to_empty(device=device)
    state_dict = torch.load("gpt2-124M.pth", map_location=device, weights_only=True)
    model.load_state_dict(state_dict, assign=True)
    del state_dict
del model; cleanup()

# 4. Memory Mapped
with track_time(timer, "Memory Mapped"):
    with torch.device("meta"):
        model = GPTModel(GPT_CONFIGS["124M"])
    state_dict = torch.load("gpt2-124M.pth", map_location=device, weights_only=True, mmap=True)
    model.load_state_dict(state_dict, assign=True)
    del state_dict
del model; cleanup()

# 5. Individual Files (requires Extension B to have been run first)
with track_time(timer, "Individual Files"):
    with torch.device("meta"):
        model = GPTModel(GPT_CONFIGS["124M"])
    model = model.to_empty(device=device)
    with torch.no_grad():
        for name, param in model.named_parameters():
            data = torch.load(
                f"gpt2-124M-params/{name}.pt",
                map_location=device, weights_only=True
            )
            param.copy_(data)
            del data
del model; cleanup()

plot_timing_comparison(timer)
print_timing_summary(timer)

### 🛑 Stop & Think -- Extension C

Look at the timing chart.

- Which technique is the **fastest**? Which is the **slowest**?
- Is the most memory-efficient technique also the fastest, or is there a trade-off?
- Where does Individual Files land on the speed spectrum?
- If you had to choose between saving memory and saving time, which would matter more in production?

<details>
<summary><b>Click for answer</b></summary>

<br>

**Memory Mapped is the fastest** -- dramatically faster than Standard and Sequential. This is because `mmap=True` lets the OS handle data loading efficiently, and `assign=True` avoids the overhead of copying tensors.

**Standard and Sequential are the slowest** -- both take significantly longer than the other techniques. Sequential's parameter-by-parameter loop doesn't help with speed, and Standard has to copy the full checkpoint into the model.

**Individual Files is mid-range** -- faster than Standard and Sequential, comparable to Meta Device. Despite loading hundreds of separate files, the small file sizes mean each read is fast. It's not the speed penalty you might expect.

Memory Mapped is **both** the most memory-efficient for GPU (tied with Sequential) **and** the fastest -- there's no trade-off here!

In production, **memory is almost always the bottleneck** -- you only load a model once, but it stays in memory the whole time. The fact that the best memory technique is also the fastest makes it an easy choice.

</details>

---

## Summary

| Technique | GPU Memory | CPU Memory | Speed | Complexity |
|-----------|-----------|-----------|-------|------------|
| **Standard** | Worst (2x) | High | Slowest | Simplest |
| **Sequential** | Best (1x) | High | Slowest | Loop required |
| **Meta Device** | Worst (2x+) | **Lowest** | Medium | `to_empty()` + `assign=True` |
| **Memory Mapped** | Best (1x) | Medium | **Fastest** | `mmap=True` + `assign=True` |
| **Individual Files** | Best (1x) | **Zero** | Medium | Many files to manage |

### The Takeaway

**Memory Mapped** is the best all-round technique -- it ties for lowest GPU memory, loads the fastest, and requires minimal code changes:

```python
# Standard approach (2x GPU memory):
state_dict = torch.load("model.pth", map_location=device, weights_only=True)
model.load_state_dict(state_dict)

# Memory-efficient approach (1x GPU memory, fastest):
with torch.device("meta"):
    model = GPTModel(config)
state_dict = torch.load("model.pth", map_location=device, weights_only=True, mmap=True)
model.load_state_dict(state_dict, assign=True)
```

`mmap=True` reads from disk on demand instead of loading everything into RAM. `assign=True` swaps tensors in directly instead of copying.

---

*COP509 -- Loughborough University*